# Advanced RAG: hybrid search and an eval harness

**Track:** RAG. **Level:** Intermediate. **CPU:** yes.

## Goals

- Compute hit rate and mean reciprocal rank on a hand-checked example before using them on a corpus.
- Fuse BM25 and dense rankings with reciprocal rank fusion.
- Score a quoted answer with a lexical faithfulness check, and show a paraphrase the check rejects.

## Sources and license

Original notebook. Hit rate and MRR are the standard ranked-retrieval metrics used in the information-retrieval textbooks and in the BEIR benchmark writeups. Reciprocal rank fusion is Cormack, Clarke, and Buettcher (2009). The faithfulness check is a lexical baseline, the same idea as the portfolio lab, on a smaller corpus. LangChain and LlamaIndex cookbooks (Apache-2.0) describe this pipeline. Their notebooks are not copied.

## Concepts

Hit rate at `k` is 1 when any gold id appears in the first `k` rows. MRR is `1 / rank` of the first gold id, and 0 when it is missing. A single query with the gold item in position 2 has hit rate at 1 equal to 0, hit rate at 2 equal to 1, and MRR equal to 0.5. Fusion by rank ignores the fact that one retriever's scores are not on the other's scale. The faithfulness check asks whether the claim's content words and numbers appear in the cited chunk. It accepts a supported quote and rejects a true sentence that was paraphrased. Both outcomes are part of the result.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'numpy'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

import math
import re
from collections import Counter

import numpy as np

TOKEN = re.compile(r"[a-z0-9]+")

def tokenize(text):
    return TOKEN.findall(text.lower())

def hit_rate(ranked, gold, k):
    return int(any(doc_id in gold for doc_id in ranked[:k]))

def mrr(ranked, gold):
    for rank, doc_id in enumerate(ranked, start=1):
        if doc_id in gold:
            return 1.0 / rank
    return 0.0

assert hit_rate(["b", "a"], {"a"}, 1) == 0
assert hit_rate(["b", "a"], {"a"}, 2) == 1
assert mrr(["b", "a", "c"], {"a"}) == 0.5
assert mrr(["c"], {"a"}) == 0.0
print("hand example: gold a is second, MRR", mrr(["b", "a", "c"], {"a"}))


## Guided exercise: fuse two rankings

BM25 is term frequency with an idf penalty. The dense vector is L2-normalized TF-IDF. One query is written with `yield` where the document says `interest`. BM25 on the raw query misses. The expanded query, and the hybrid list, hit.


In [ ]:
DOCS = {
    "savings": "The savings account yields 3 percent annual interest and has no monthly fee.",
    "card": "The credit card has an annual fee of 49 dollars.",
    "wire": "An international wire costs 15 dollars.",
    "loan": "Personal loans carry a fixed interest rate of 7 percent.",
}
GLOSSARY = {"yield": "interest", "yearly": "annual"}

def expand(query):
    extra = [GLOSSARY[tok] for tok in tokenize(query) if tok in GLOSSARY]
    return query if not extra else query + " " + " ".join(extra)

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.ids = list(docs)
        self.tokenized = [tokenize(docs[i]) for i in self.ids]
        self.df = Counter(term for doc in self.tokenized for term in set(doc))
        self.tf = [Counter(doc) for doc in self.tokenized]
        self.lengths = [len(doc) or 1 for doc in self.tokenized]
        self.avgdl = sum(self.lengths) / len(self.lengths)
        self.k1, self.b = k1, b

    def rank(self, query, k=4):
        scored = []
        q = tokenize(query)
        for i, doc_id in enumerate(self.ids):
            score = 0.0
            for term in q:
                freq = self.tf[i].get(term, 0)
                if not freq:
                    continue
                idf = math.log(1 + (len(self.ids) - self.df[term] + 0.5) / (self.df[term] + 0.5))
                denom = freq + self.k1 * (1 - self.b + self.b * self.lengths[i] / self.avgdl)
                score += idf * freq * (self.k1 + 1) / denom
            if score > 0:
                scored.append((doc_id, score))
        scored.sort(key=lambda item: -item[1])
        return scored[:k]

class Dense:
    def __init__(self, docs):
        self.ids = list(docs)
        self.texts = [docs[i] for i in self.ids]
        tokenized = [tokenize(text) for text in self.texts]
        df = Counter(term for doc in tokenized for term in set(doc))
        self.idf = {t: math.log((1 + len(self.ids)) / (1 + df[t])) + 1 for t in df}
        self.vocab = {t: i for i, t in enumerate(sorted(self.idf))}
        self.matrix = self._embed(self.texts)

    def _embed(self, texts):
        m = np.zeros((len(texts), len(self.vocab)))
        for r, text in enumerate(texts):
            for term, count in Counter(tokenize(text)).items():
                j = self.vocab.get(term)
                if j is not None:
                    m[r, j] = count * self.idf[term]
            n = np.linalg.norm(m[r])
            if n:
                m[r] /= n
        return m

    def rank(self, query, k=4):
        v = self._embed([query])[0]
        if np.linalg.norm(v) == 0:
            return []
        scores = self.matrix @ v
        order = np.argsort(-scores)
        return [(self.ids[i], float(scores[i])) for i in order if scores[i] > 0][:k]

def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return [doc_id for doc_id, _ in sorted(scores.items(), key=lambda item: -item[1])]

bm25, dense = BM25(DOCS), Dense(DOCS)
QUESTIONS = [
    ("savings account annual interest", {"savings"}),
    ("credit card annual fee", {"card"}),
    ("international wire cost", {"wire"}),
    ("yearly yield", {"savings"}),
]
rows = []
for query, gold in QUESTIONS:
    expanded = expand(query)
    sparse = [doc for doc, _ in bm25.rank(expanded)]
    neural = [doc for doc, _ in dense.rank(expanded)]
    fused = rrf([sparse, neural])
    raw = [doc for doc, _ in bm25.rank(query)]
    rows.append((query, hit_rate(raw, gold, 3), hit_rate(fused, gold, 3), mrr(fused, gold), fused[:2]))
for row in rows:
    print(row)
assert rows[0][2] == 1 and rows[3][1] == 0 and rows[3][2] == 1
mean_mrr = sum(row[3] for row in rows) / len(rows)
print("mean hybrid MRR", round(mean_mrr, 3))
assert mean_mrr > 0.5


## Your turn

The synonym query is the last row. Confirm the raw BM25 hit rate is 0 and the fused hit rate is 1.

## Solution


In [ ]:
query, raw_hit, fused_hit, reciprocal, top = rows[-1]
print(query, "raw", raw_hit, "fused", fused_hit, "mrr", reciprocal, top)
assert raw_hit == 0 and fused_hit == 1


## Beyond the course

Quote the savings sentence and cite it. The lexical check passes. A paraphrase that preserves the 3 percent fact and shares almost no content words fails the same check. Report both. The failure is the reason a later faithfulness model has to understand the sentence rather than its tokens.

The check also rejects a claim whose number is not in the chunk.


In [ ]:
STOP = {"the", "a", "an", "of", "and", "to", "is", "in", "on", "has", "no"}

def supported(claim, source, minimum=0.4):
    if set(re.findall(r"\d+", claim)) - set(re.findall(r"\d+", source)):
        return False
    claim_terms = set(tokenize(claim)) - STOP
    source_terms = set(tokenize(source)) - STOP
    if not claim_terms:
        return False
    return len(claim_terms & source_terms) / len(claim_terms) >= minimum

source = DOCS["savings"]
quote = "The savings account yields 3 percent annual interest."
paraphrase = "Cash in that product earns three percent a year."
wrong_number = "The savings account yields 9 percent annual interest."
print("quote", supported(quote, source), "paraphrase", supported(paraphrase, source), "wrong number", supported(wrong_number, source))
assert supported(quote, source)
assert not supported(paraphrase, source)
assert not supported(wrong_number, source)
print("LAB OK")


## Research questions

1. MRR treats the second position as half a point. For a user who only reads the top hit, which of hit rate at 1 or MRR matches the product?
2. Rank fusion recovered the synonym query because of the glossary, not because of the dense geometry. What would you change in the embedder before removing the glossary?
3. The paraphrase was true and was rejected. Construct a false sentence this check would accept. The portfolio lab asks the same question of a larger corpus.

## Further reading

- Cormack, Clarke, and Buettcher, *Reciprocal Rank Fusion* (SIGIR 2009).
- Thakur et al., *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval* (2021).
- Lewis et al., RAG (2020). Nogueira and Cho, passage reranking with BERT (2019).
- LangChain and LlamaIndex retrieval docs (Apache-2.0). Not copied.
